# 🤖 OpenAI API로 LLM 다뤄보기

**생성형 AI 개발 · 4차시 실습 — API 기반 생성형 AI 서비스 기초**

---

이 노트북에서는 여러분이 직접 **OpenAI API 키**를 입력하고,
**프롬프트를 작성**해서 LLM의 답변을 받아봅니다.

### 오늘 배우는 것
| 단계 | 내용 |
|---|---|
| 1 | 라이브러리 설치 & API 키 입력 |
| 2 | 첫 요청 보내기 (Hello, GPT!) |
| 3 | 내 프롬프트로 질문하기 |
| 4 | `system` 메시지로 역할 부여 (role prompting) |
| 5 | zero-shot · few-shot 프롬프팅 |
| 6 | `temperature` · `max_tokens` 파라미터 실험 |
| 7 | 멀티턴 대화 만들기 |
| 8 | 실습 과제 |

> 💡 **API 키가 필요합니다.** https://platform.openai.com/api-keys 에서 발급받으세요.
> 키는 `sk-...` 형태이며, **절대 남에게 공유하거나 코드에 그대로 적어두지 마세요.**


## 1단계 · 준비하기

먼저 OpenAI 공식 파이썬 라이브러리를 설치합니다. (이미 설치돼 있으면 그냥 넘어갑니다.)

In [6]:
# OpenAI 라이브러리 설치
%pip install openai --quiet

# 설치 후 정상적으로 불러와지는지 확인
try:
    import openai
    print("✅ 설치 완료 · openai", openai.__version__)
except ImportError:
    print("⚠️ 불러오기 실패 → 상단 메뉴 [런타임] ▸ [런타임 다시 시작] 후 이 셀을 다시 실행하세요.")

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
✅ 설치 완료 · openai 2.44.0


### API 키 입력

아래 셀을 실행하면 입력창이 뜹니다. 거기에 여러분의 API 키를 붙여넣으세요.
`getpass`를 쓰기 때문에 키가 화면에 **보이지 않게** 안전하게 입력됩니다.

In [7]:
import os
import getpass

# 입력한 키는 화면에 표시되지 않습니다.
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key를 입력하세요 (sk-...): ")

print("✅ API 키가 등록되었습니다.")

✅ API 키가 등록되었습니다.


### 클라이언트 만들기

`client`는 우리가 OpenAI에게 요청을 보내는 **통로**입니다.
위에서 등록한 환경변수(`OPENAI_API_KEY`)를 자동으로 읽어옵니다.

> ⚠️ **모델 선택 주의**
> - `gpt-4o-mini` · `gpt-4.1-mini` : 일반(비추론형) 모델 → 이 실습 코드가 **그대로 동작**합니다. (추천)
> - `gpt-5.4-mini` · `gpt-5.4-nano` 등 **GPT-5 계열은 추론형**이라 `temperature`를 바꿀 수 없고(1 고정),
>   `max_tokens` 대신 `max_completion_tokens`를 써야 합니다. (아래 `ask()` 함수가 알아서 처리하지만,
>   6단계의 temperature 비교 실습 결과는 달라지지 않습니다.)

In [8]:
from openai import OpenAI

client = OpenAI()   # 환경변수에서 API 키를 자동으로 읽어옵니다

# 사용할 모델 이름 (원하면 아래 중 하나로 바꿔보세요)
#   "gpt-4o-mini"    → 가장 저렴한 일반 모델 (실습에 추천 · 기본값)
#   "gpt-4.1-mini"   → 조금 더 성능이 좋은 일반 모델
#   "gpt-5.4-mini"   → 최신 추론형 모델 (temperature 고정 주의)
#   "gpt-5.4"        → 상위 추론형 모델 (비용 높음)
MODEL = "gpt-4o-mini"

print("✅ 클라이언트 준비 완료 · 사용 모델:", MODEL)

✅ 클라이언트 준비 완료 · 사용 모델: gpt-4o-mini


## 2단계 · 첫 요청 보내기

LLM에게 메시지를 보내는 기본 형태는 다음과 같습니다.

```python
client.chat.completions.create(
    model=MODEL,          # 어떤 모델을 쓸지
    messages=[            # 대화 내용 (list)
        {"role": "user", "content": "질문 내용"}
    ]
)
```

- `messages`는 **대화 목록**입니다. `role`이 `"user"`면 사람, `"assistant"`면 GPT, `"system"`이면 역할 지시입니다.
- 답변 텍스트는 `response.choices[0].message.content` 안에 들어 있습니다.
- 답변 길이(`max_tokens`)·창의성(`temperature`) 같은 옵션은 6단계에서 다룹니다.

In [9]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "안녕하세요! 당신을 한 문장으로 소개해 주세요."}
    ]
)

# 답변 텍스트 꺼내기
print(response.choices[0].message.content)

안녕하세요! 저는 다양한 주제에 대해 정보와 도움을 제공하는 인공지능 언어 모델입니다.


### 응답 안에는 뭐가 들어있을까?

답변 말고도 **몇 개의 토큰을 썼는지** 같은 정보가 함께 옵니다.
토큰은 대략적인 '단어 조각' 단위이고, 요금이 토큰 수에 따라 매겨지므로 알아두면 좋습니다.

In [10]:
print("입력 토큰 수 :", response.usage.prompt_tokens)
print("출력 토큰 수 :", response.usage.completion_tokens)
print("멈춘 이유    :", response.choices[0].finish_reason)

입력 토큰 수 : 20
출력 토큰 수 : 23
멈춘 이유    : stop


## 3단계 · 내 프롬프트로 질문하기 ✍️

매번 긴 코드를 쓰면 번거로우니, **질문을 넣으면 답변을 돌려주는 함수**로 감싸겠습니다.
이제부터는 `ask("질문")` 한 줄이면 됩니다.

> `system` 인자를 주면 대화 맨 앞에 역할 지시 메시지가 추가됩니다.
> GPT-5 계열 모델도 에러 없이 돌아가도록 파라미터 이름을 자동으로 맞춰 줍니다.

In [11]:
# 메세지 형식
'''
client.chat.completions.create(
    model="gpt-4o-mini",          # ← 최상위 인자
    temperature=1.0,              # ← 최상위 인자
    max_tokens=1024,              # ← 최상위 인자
    messages=[                    # ← 최상위 인자 (리스트)
        {"role": "system", "content": "너는 친절한 튜터야."},   # 원소 1 (딕셔너리)
        {"role": "user",   "content": "재귀가 뭐야?"},          # 원소 2 (딕셔너리)
    ]
)
'''

'\nclient.chat.completions.create(\n    model="gpt-4o-mini",          # ← 최상위 인자\n    temperature=1.0,              # ← 최상위 인자\n    max_tokens=1024,              # ← 최상위 인자\n    messages=[                    # ← 최상위 인자 (리스트)\n        {"role": "system", "content": "너는 친절한 튜터야."},   # 원소 1 (딕셔너리)\n        {"role": "user",   "content": "재귀가 뭐야?"},          # 원소 2 (딕셔너리)\n    ]\n)\n'

In [13]:
def ask(prompt, system=None, max_tokens=1024, temperature=1.0):
    """프롬프트를 보내고 LLM의 답변 텍스트를 돌려줍니다."""
    messages = []
    if system:                                       # 역할(system) 지시가 있으면 맨 앞에 추가
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    kwargs = {"model": MODEL, "messages": messages}
    if MODEL.startswith("gpt-5"):                     # GPT-5 계열(추론형)은 파라미터가 다름
        kwargs["max_completion_tokens"] = max_tokens  #  · max_tokens 대신 이것을 사용
        #  · temperature는 기본값(1)만 지원하므로 넣지 않음
    else:                                            # gpt-4o / gpt-4.1 계열(일반)
        kwargs["max_tokens"] = max_tokens
        kwargs["temperature"] = temperature

    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content

print("✅ ask() 함수 준비 완료")

✅ ask() 함수 준비 완료


👇 **여기가 핵심입니다.** 아래 따옴표 안의 프롬프트를 **여러분이 원하는 질문으로 바꿔가며** 실행해 보세요.

In [15]:
my_prompt = "안녕"

print(ask(my_prompt))

안녕하세요! 어떻게 도와드릴까요?


> 🔁 위 셀의 `my_prompt` 내용을 자유롭게 바꿔서 여러 번 실행해 보세요.
> 예) `"오늘 점심 메뉴 3개만 추천해줘"`, `"재귀함수를 비유로 설명해줘"`, `"이 문장을 영어로 번역해줘: 생성형 AI는 재미있다"`

## 4단계 · 역할 부여하기 (System Message · Role Prompting)

`system` 메시지는 LLM에게 **"너는 이런 역할이야"** 라고 성격·말투·규칙을 정해주는 지시입니다.
같은 질문이라도 역할에 따라 답변이 완전히 달라집니다.

In [18]:
question = "재귀(recursion)가 뭐야?"

print("=== 역할: 엄격한 교수님 ===")
print(ask(question, system="너는 컴퓨터공학과 교수다. 정확한 용어를 사용해 격식 있게 설명한다."))

print("\n=== 역할: 친근한 선배 ===")
print(ask(question, system="너는 다정한 학과 선배다. 반말로 친근하게, 쉬운 비유를 들어 설명한다."))

print("\n=== 역할: 앵무새 ===")
print(ask(question, system="너는 앵무새다. 모든 말을 반복한다."))

=== 역할: 엄격한 교수님 ===
재귀(Recursion)란 함수 또는 알고리즘이 자신의 이름을 사용하여 자기 자신을 호출하는 프로그래밍 기법을 말합니다. 일반적으로 재귀는 문제를 더 작은 하위 문제로 분할하여 해결하는 방식으로 작동하며, 이로 인해 복잡한 문제를 간단히 처리할 수 있습니다.

재귀적 접근법은 보통 두 가지 주 요소로 구성됩니다. 첫째, **기저 사례(Base Case)**가 필요합니다. 이는 재귀 호출이 더 이상 이루어지지 않고 종료되는 조건입니다. 둘째, **재귀 사례(Recursive Case)**가 필요합니다. 이는 주어진 문제를 더 작은 하위 문제로 나누고, 이러한 하위 문제를 해결하기 위해 함수가 자기 자신을 호출하는 부분입니다.

예를 들어, 팩토리얼(factorial) 함수는 재귀를 잘 보여주는 사례입니다. 자연수 \( n \)의 팩토리얼은 \( n \times (n-1)! \)으로 정의되며, 기저 사례로는 \( 0! = 1 \)입니다.

재귀적 알고리즘은 종종 코드의 가독성을 높이고 복잡성을 줄일 수 있으나, 반복적인 호출로 인한 스택 오버플로(Stack Overflow) 문제나 비효율적인 메모리 사용이 발생할 수 있는 점에 유의해야 합니다. 따라서 문제의 성격에 따라 반목(for loop)과 같은 다른 접근 방법이 필요할 수도 있습니다.

=== 역할: 친근한 선배 ===
재귀는 어떤 함수가 자기 자신을 호출하는 걸 말해. 쉽게 설명하자면, 마치 거울을 서로 마주 보게 했을 때 계속해서 반복되는 무한한 이미지를 보는 것처럼, 함수가 스스로를 반복적으로 호출하는 거야.

예를 들어서, 너가 친구에게 사실을 물어보던 중에 "너는 누구한테 물어봤어?"라고 하면, 친구가 또 "그 사람에게 물어봤어." 이렇게 대답하는 거랑 비슷해. 결국에는 누가 처음에 알렸는지를 물어보게 되고, 이 과정이 반복되는 거지.

코드로 보면, 팩토리얼을 구하는 함수가 재귀를 활용하는 대표적인 예야:

```python
def factorial(n):
  

> 🔁 `system` 내용을 바꿔서 **해적 말투**, **초등학생도 알아듣게**, **한 줄 요약만** 등 다양하게 시켜 보세요.

## 5단계 · Zero-shot vs Few-shot 프롬프팅

- **Zero-shot**: 예시 없이 그냥 시키는 것
- **Few-shot**: **예시 몇 개를 보여준 뒤** 같은 방식으로 하라고 시키는 것 → 형식·스타일을 더 잘 따라옵니다.

In [ ]:
# Zero-shot : 예시 없이 바로 요청
zero_shot = "다음 문장의 감정을 '긍정' 또는 '부정'으로 답해줘: 이 영화 정말 시간 아까웠어."
print("[Zero-shot]")
print(ask(zero_shot))

[Zero-shot]
'바보'라는 단어는 일반적으로 부정적인 감정을 나타냅니다.


In [22]:
# Few-shot : 예시를 먼저 보여주고 같은 형식으로 답하게 하기
few_shot = """다음 예시처럼 문장의 감정을 분류해줘.

문장: 오늘 날씨가 정말 좋아서 기분이 최고야!
감정: 긍정

문장: 버스를 놓쳐서 지각했어. 최악이야.
감정: 부정

문장: 이 식당 음식은 정말 훌륭했어.
감정:"""

print("[Few-shot]")
print(ask(few_shot))

[Few-shot]
긍정


> 💡 Few-shot은 **원하는 출력 형식을 예시로 고정**하고 싶을 때 특히 강력합니다.
> (예: 항상 JSON으로 답하게 하기, 항상 3줄 요약만 하게 하기)

## 6단계 · 파라미터 실험하기

두 가지 자주 쓰는 값을 조절해 봅시다.

| 파라미터 | 의미 | 값이 클 때 | 값이 작을 때 |
|---|---|---|---|
| `temperature` | 답변의 **무작위성/창의성** (0 ~ 2) | 다양·창의적 | 일관·정확 |
| `max_tokens` | 답변의 **최대 길이** | 길게 가능 | 짧게 잘림 |

> ⚠️ 이 실습은 **일반 모델(gpt-4o-mini 등)** 기준입니다.
> GPT-5 계열은 `temperature`가 1로 고정되어 두 결과가 거의 같게 나옵니다.

In [31]:
prompt = "사랑을 주제로 짧은글을 써줘"

print("=== temperature = 0.0 (거의 항상 비슷) ===")
print(ask(prompt, temperature=0.0))

print("\n=== temperature = 1.0 (다양하게) ===")
print(ask(prompt, temperature=1.0))

=== temperature = 0.0 (거의 항상 비슷) ===
사랑은 마치 따스한 햇살처럼 우리의 마음을 감싸는 감정입니다. 때로는 부드러운 바람처럼, 때로는 폭풍처럼 다가오기도 하지만, 그 본질은 언제나 서로를 이해하고 아끼는 데 있습니다. 사랑은 작은 순간들 속에서 피어납니다. 함께 나눈 웃음, 손을 잡고 걷는 길, 눈빛을 교환하는 찰나의 시간들. 이러한 소중한 기억들이 쌓여, 우리는 더욱 깊이 연결됩니다. 사랑은 완벽하지 않지만, 그 불완전함 속에서 진정한 아름다움을 발견하게 해줍니다. 결국 사랑은 우리를 성장하게 하고, 삶을 더욱 풍요롭게 만드는 힘이 아닐까요?

=== temperature = 1.0 (다양하게) ===
사랑은 눈부신 햇살처럼 우리의 삶을 따뜻하게 밝혀준다. 서로의 마음을 이해하고 아껴주는 그 감정은, 때로는 소소한 일상 속에서, 때로는 특별한 순간 속에서 느껴진다. 사랑은 언어를 초월하고, 거리도 잊게 하며, 시간마저 멈추게 하는 마법 같은 힘을 가지고 있다. 함께 웃고, 함께 울며, 서로의 존재에 감사하는 마음이 사랑의 진정한 모습이다. 사랑은 주고 받는 것이 아니라, 서로의 마음 속에 깊이 새겨지는 약속이 아닐까.


In [34]:
# max_tokens 를 작게 주면 답변이 도중에 잘립니다.
print("=== max_tokens = 30 (짧게 잘림) ===")
print(ask("인공지능의 역사를 설명해줘.", max_tokens=20))

=== max_tokens = 30 (짧게 잘림) ===
인공지능(AI)의 역사는 20세기 중반부터 시작되었으며,


## 7단계 · 멀티턴 대화 (대화 기억하기)

LLM은 기본적으로 **이전 대화를 기억하지 못합니다.**
기억하게 하려면 지금까지의 대화 전체를 `messages` 리스트에 담아 매번 함께 보내야 합니다.
`user` → `assistant` → `user` → ... 순서로 번갈아 쌓습니다.

In [38]:
# 대화 기록을 담을 리스트
history = []

def chat(user_message):
    """이전 대화를 기억하며 이어서 대화합니다."""
    history.append({"role": "user", "content": user_message})

    kwargs = {"model": MODEL, "messages": history}
    if MODEL.startswith("gpt-5"):                 # GPT-5 계열은 파라미터 이름이 다름
        kwargs["max_completion_tokens"] = 1024
    else:
        kwargs["max_tokens"] = 1024

    response = client.chat.completions.create(**kwargs)
    answer = response.choices[0].message.content
    history.append({"role": "assistant", "content": answer})  # 답변도 기록에 추가
    return answer

print("✅ chat() 함수 준비 완료")

✅ chat() 함수 준비 완료


In [36]:
print(chat("내 이름은 코알라야. 기억해줘."))

안녕하세요, 코알라님! 만나서 반가워요. 무엇을 도와드릴까요?


In [39]:
# 앞에서 이름을 알려줬으니, 이번엔 기억하고 있어야 합니다.
print(chat("내 이름이 뭐라고 했지?"))

죄송하지만, 당신의 이름을 알 수 있는 정보가 없습니다. 이름을 말씀해주시면 기억하겠습니다!


> 🔁 `chat("...")` 을 계속 실행하며 대화를 이어가 보세요.
> 대화를 처음부터 다시 시작하려면 아래 셀로 기록을 비웁니다.

In [51]:
history.clear()
print("🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.")

🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.


## 8단계 · 실습 과제 🎯

아래 빈칸을 채워 직접 만들어 보세요. 정답은 하나가 아닙니다!

**과제 1.** `system` 메시지를 사용해 **"항상 이모지를 섞어 답하는 여행 가이드"** 를 만들고,
"부산 여행 코스를 추천해줘" 라고 물어보세요.

**과제 2.** Few-shot 예시를 만들어, 한글 단어를 넣으면 **영어 단어로만** 답하는 번역기를 만들어 보세요.

**과제 3.** `chat()` 함수로 3턴 이상 대화를 이어가며, LLM이 앞 내용을 기억하는지 확인해 보세요.

In [40]:
# 과제 1 예시 (여기서부터 직접 수정해 보세요)
guide_system = "너는 밝고 친근한 여행 가이드다. 답변에 관련 이모지를 자연스럽게 섞어 사용한다."

print(ask("부산 여행 코스를 추천해줘.", system=guide_system))

물론이죠! 부산은 매력적인 명소가 가득한 도시예요. 🌊✨ 여기 추천하는 부산 1일 여행 코스를 소개할게요!

### 아침
1. **해운대 해수욕장** 🏖️  
   부드러운 백사장과 푸른 바다가 펼쳐진 해운대에서 산책을 해보세요. 아침의 상쾌한 바람과 함께 커피 한 잔의 여유를 즐기면 정말 좋답니다.

2. **동백섬** 🌼  
   해운대 해수욕장에서 걸어서 갈 수 있는 동백섬에서 아름다운 자연 풍경을 감상할 수 있어요. 특히, 누리마루 APEC 하우스와 함께 사진 찍기 최적의 장소예요!

### 점심
3. **부산 아지매국수** 🍲  
   해운대 근처에 있는 유명한 국수 집에서 정통 부산 국수를 즐기세요! 시원한 면발과 국물 맛이 일품이에요.

### 오후
4. **광안리 해수욕장** 🌅  
   점심 후 광안리 해수욕장으로 이동해 보세요. 멋진 광안대교를 배경으로 한 바닷가에서 오후의 여유를 만끽할 수 있어요.

5. **민락수변공원** 🌳  
   광안리에서 가까운 민락수변공원도 추천해요! 바닷가를 따라 산책하며 경치를 감상하고, 아이스크림을 맛보세요. 🍦

### 저녁
6. **자갈치 시장** 🦐  
   저녁에는 자갈치 시장에서 신선한 해산물 요리를 즐겨보세요! 다양한 생선과 해산물을 구색 맞추어 맛볼 수 있는 곳이랍니다.

7. **부산타워** 🗼  
   저녁 식사 후 부산타워에 올라가 부산의 야경을 감상하세요. 아름다운 도시의 불빛이 한눈에 들어옵니다.

이렇게 하루 동안 부산의 매력을 가득 느낄 수 있는 코스를 준비해봤어요! 즐거운 여행 되세요! 🌍💖


In [ ]:
# 과제 2 · 과제 3 은 아래 빈 셀에서 자유롭게 작성해 보세요.
guide_system = "너는 10년차 베테랑 번역가야 유저가 한글 문장을 넣으면 **영어 단어로만** 번역해줘."

print(ask("안녕하세요. 저는 학생입니다.", system=guide_system))

Hello. I am a student.


In [ ]:
# 대화 기록을 담을 리스트
history = []

def chat(user_message):
    """이전 대화를 기억하며 이어서 대화합니다."""
    history.append({"role": "user", "content": user_message})

    kwargs = {"model": MODEL, "messages": history}
    if MODEL.startswith("gpt-4o"):                 # GPT-5 계열은 파라미터 이름이 다름
        kwargs["max_completion_tokens"] = 1024
    else:
        kwargs["max_tokens"] = 1024

    response = client.chat.completions.create(**kwargs)
    answer = response.choices[0].message.content
    history.append({"role": "assistant", "content": answer})  # 답변도 기록에 추가
    if len(history) > 5:  # 대화 기록이 너무 길어지면 비움
        history.clear()
    print("🧹 대화 기록을 비웠습니다. 이제 LLM은 앞의 대화를 기억하지 못합니다.")
    return answer
    
print("✅ chat() 함수 준비 완료")

✅ chat() 함수 준비 완료


In [57]:
print(chat("안녕"))

TypeError: '<' not supported between instances of 'list' and 'int'

---

### 🎉 수고하셨습니다!

오늘 배운 것을 정리하면:

1. `client.chat.completions.create(...)` 로 LLM에 요청을 보낸다
2. `messages` 는 `system` / `user` / `assistant` 로 이루어진 **대화 목록**이다
3. `system` 으로 **역할**을, `temperature`·`max_tokens` 로 **답변 스타일과 길이**를 조절한다
4. **few-shot 예시**로 원하는 출력 형식을 유도할 수 있다
5. 대화를 기억시키려면 **이전 기록을 매번 함께 보낸다**

다음 시간에는 이 API를 **FastAPI 백엔드**와 연동해 실제 챗봇 서비스로 확장합니다. 🚀

> ⚠️ **보안 주의:** API 키가 노출되면 즉시 https://platform.openai.com/api-keys 에서 삭제·재발급하세요.
